# <span style="color:#7C3AED">Testing a GPU Transformer Against the Classical Model</span>

<table width="100%" cellpadding="10" cellspacing="0">
<tr bgcolor="#f8fafc"><td>
<p><b>Fundamentals of Natural Language Processing</b> | Universitat Autonoma de Barcelona | 2025 2026</p>
<p>Phoebe Iglesias (1713459), David Redrejo (1790336), Pau Rossell (1750424)</p>
<h3><font color="#7C3AED">Notebook question</font></h3>
<p>Can a Spanish biomedical transformer beat our tuned character SVM on very short literals?</p>
<h3><font color="#7C3AED">Connection with the previous notebook</font></h3>
<p>This follows <b><font color="#16A34A">Notebook 03: Tuning the Baseline Until It Generalizes</font></b>. The tuned SVM is now strong enough that RoBERTa has a real benchmark to beat.</p>
<h3><font color="#7C3AED">What this chapter contributes</font></h3>
<p>We prepare a GPU ready transformer experiment with tokenization, pooling, early stopping, and submission checks.</p>
</td></tr>
</table>

<table width="100%" cellpadding="8" cellspacing="0">
<tr bgcolor="#7C3AED"><th><font color="white">Move</font></th><th><font color="white">What we try to understand</font></th></tr>
<tr><td>1</td><td>Setup GPU</td></tr><tr><td>2</td><td>Load text</td></tr><tr><td>3</td><td>Build target</td></tr><tr><td>4</td><td>Tokenize</td></tr><tr><td>5</td><td>Encode</td></tr><tr><td>6</td><td>Train</td></tr><tr><td>7</td><td>Inspect</td></tr><tr><td>8</td><td>Submit</td></tr>
</table>

## Chapter Map

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

chapter_color = "#7C3AED"
labels = ['Setup GPU', 'Load text', 'Build target', 'Tokenize', 'Encode', 'Train', 'Inspect', 'Submit']

fig, ax = plt.subplots(figsize=(14, 2.6))
ax.set_xlim(0, len(labels))
ax.set_ylim(0, 1)
ax.axis("off")

for idx, label in enumerate(labels):
    card = FancyBboxPatch(
        (idx + 0.06, 0.25), 0.88, 0.48,
        boxstyle="round,pad=0.04,rounding_size=0.05",
        linewidth=1.4,
        edgecolor=chapter_color,
        facecolor="#f8fafc"
    )
    ax.add_patch(card)
    ax.text(idx + 0.5, 0.49, label, ha="center", va="center", fontsize=10.5, color="#0f172a", wrap=True)
    if idx < len(labels) - 1:
        ax.annotate("", xy=(idx + 1.02, 0.49), xytext=(idx + 0.94, 0.49), arrowprops=dict(arrowstyle=">", color=chapter_color, lw=1.8))

ax.text(0.02, 0.9, "How this notebook moves", fontsize=14, weight="bold", color=chapter_color)
plt.show()

## Setup Paths and GPU

We start by detecting the device and setting the training configuration. This notebook is designed for GPU execution, although it can still run slowly on CPU.

In [ ]:
import os
import sys
import time
import random
import warnings
from pathlib import Path
import re

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / "src").exists() else NOTEBOOK_DIR.parent
DATA_DIR = PROJECT_ROOT / "data"
SUBMISSION_DIR = PROJECT_ROOT / "submissions"
MODELS_DIR = PROJECT_ROOT / "models"
SUBMISSION_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from data_processing import extract_category
from evaluation import generate_submission

required_files = [
    DATA_DIR / "codification_data.csv",
    DATA_DIR / "leaderboard_data.csv",
]
missing = [str(path) for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing required data files. Place the project CSV files in the data/ folder: "
        + ", ".join(missing)
    )

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"

MODEL_NAME = "PlanTL-GOB-ES/roberta-base-biomedical-clinical-es"
MAX_LEN = 64
BATCH_SIZE = 32 if device.type == "cuda" else 8
EPOCHS = 8
PATIENCE = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

print(f"Project root : {PROJECT_ROOT}")
print(f"Device       : {device}")
if device.type == "cuda":
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
    print(f"CUDA memory  : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print(f"Mixed precision: {use_amp}")
print(f"Batch size      : {BATCH_SIZE}")

## Minimal Cleanup Data Loading

Unlike the TF IDF notebooks, we do not strip the text heavily. RoBERTa has its own tokenizer, so we preserve more of the original form and only collapse whitespace.

In [ ]:
def preprocess_dl(text):
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text

codif_df = pd.read_csv(DATA_DIR / "codification_data.csv")
lead_df = pd.read_csv(DATA_DIR / "leaderboard_data.csv")

codif_df["Literal"] = codif_df["Literal"].apply(preprocess_dl)
lead_df["Literal"] = lead_df["Literal"].apply(preprocess_dl)

print(f"Training rows  : {len(codif_df):,}")
print(f"Unique literals: {codif_df['Literal'].nunique():,}")
print(f"Unique codes   : {codif_df['Code'].nunique():,}")
print(f"Leaderboard    : {len(lead_df):,}")

display(codif_df.head())
display(lead_df.head())

assert {"Code", "Literal"}.issubset(codif_df.columns)
assert {"id", "Literal"}.issubset(lead_df.columns)

## Category Per Literal

To keep the comparison fair, we build the same target as before: one first character category per literal. The deep model changes the representation, not the task definition.

In [ ]:
work_df = codif_df.copy()
work_df["y_category"] = work_df["Code"].apply(extract_category)

ambiguity = (
    work_df.groupby("Literal")["y_category"]
    .nunique()
    .reset_index(name="n_categories")
)
ambiguous_literals = ambiguity[ambiguity["n_categories"] > 1]

cat_df = (
    work_df.groupby("Literal")["y_category"]
    .agg(lambda s: s.value_counts().index[0])
    .reset_index()
)

categories = sorted(cat_df["y_category"].unique())
cat2idx = {cat: idx for idx, cat in enumerate(categories)}
idx2cat = {idx: cat for cat, idx in cat2idx.items()}
cat_df["label"] = cat_df["y_category"].map(cat2idx)

X_train, X_val, y_train, y_val = train_test_split(
    cat_df["Literal"].values,
    cat_df["label"].values,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=cat_df["label"].values,
)

print(f"Rows after one-label-per-literal: {len(cat_df):,}")
print(f"Categories                      : {len(categories)}")
print(f"Ambiguous original literals     : {len(ambiguous_literals):,}")
print(f"Train size                      : {len(X_train):,}")
print(f"Validation size                 : {len(X_val):,}")

display(cat_df.head(10))

## Tokenizer and DataLoaders

Now the text becomes tensors. The tokenizer creates input ids and attention masks, and the DataLoaders prepare batches for training and validation.

In [ ]:
class ICD10Dataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=64):
        self.texts = list(texts)
        self.labels = None if labels is None else list(labels)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        encoding = self.tokenizer(
            str(self.texts[index]),
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding="max_length",
            truncation=True,
            return_attention_mask=True,
            return_tensors="pt",
        )
        sample = {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
        }
        if self.labels is not None:
            sample["labels"] = torch.tensor(self.labels[index], dtype=torch.long)
        return sample

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_dataset = ICD10Dataset(X_train, y_train, tokenizer, max_len=MAX_LEN)
val_dataset = ICD10Dataset(X_val, y_val, tokenizer, max_len=MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

example_batch = next(iter(train_loader))
print(f"Input batch shape: {example_batch['input_ids'].shape}")
print(f"Number of train batches: {len(train_loader)}")
print(f"Number of val batches  : {len(val_loader)}")

## RoBERTa Architecture

The model uses a biomedical RoBERTa encoder, mean pooling over non padding tokens, dropout, and a linear classifier. Mean pooling makes sense here because the whole literal is usually short enough to matter.

In [ ]:
class RobertaMeanPoolingClassifier(nn.Module):
    def __init__(self, model_name, num_classes):
        super().__init__()
        self.roberta = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.roberta.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        hidden = outputs.last_hidden_state
        mask = attention_mask.unsqueeze(-1).expand(hidden.size()).float()
        pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
        return self.classifier(self.dropout(pooled))

model = RobertaMeanPoolingClassifier(MODEL_NAME, num_classes=len(categories)).to(device)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## Training With Early Stopping

We train with AdamW, cross entropy, a scheduler, mixed precision when CUDA is available, and early stopping. This keeps the GPU experiment controlled instead of just large.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=max(1, int(0.1 * total_steps)),
    num_training_steps=total_steps,
)
loss_fn = nn.CrossEntropyLoss()
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

best_acc = 0.0
epochs_without_improvement = 0
best_model_path = MODELS_DIR / "best_roberta_baseline.pt"
training_history = []

for epoch in range(1, EPOCHS + 1):
    start = time.time()
    model.train()
    train_loss = 0.0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=use_amp):
            logits = model(input_ids, attention_mask)
            loss = loss_fn(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        train_loss += loss.item()

    model.eval()
    val_loss = 0.0
    preds = []
    gold = []

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            with torch.cuda.amp.autocast(enabled=use_amp):
                logits = model(input_ids, attention_mask)
                loss = loss_fn(logits, labels)

            val_loss += loss.item()
            preds.extend(logits.argmax(dim=1).cpu().numpy())
            gold.extend(labels.cpu().numpy())

    train_loss_avg = train_loss / max(len(train_loader), 1)
    val_loss_avg = val_loss / max(len(val_loader), 1)
    val_acc = accuracy_score(gold, preds)
    val_weighted_f1 = f1_score(gold, preds, average="weighted", zero_division=0)

    training_history.append({
        "epoch": epoch,
        "train_loss": train_loss_avg,
        "val_loss": val_loss_avg,
        "val_accuracy": val_acc,
        "val_weighted_f1": val_weighted_f1,
        "seconds": time.time() - start,
    })

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train_loss={train_loss_avg:.4f} | "
        f"val_loss={val_loss_avg:.4f} | "
        f"val_acc={val_acc:.4f} | "
        f"val_weighted_f1={val_weighted_f1:.4f} | "
        f"{time.time() - start:.1f}s"
    )

    if val_acc > best_acc:
        best_acc = val_acc
        epochs_without_improvement = 0
        torch.save(model.state_dict(), best_model_path)
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping after epoch {epoch}.")
        break

history_df = pd.DataFrame(training_history)
print(f"Best validation accuracy: {best_acc:.4f}")
display(history_df)

assert best_model_path.exists()
assert len(history_df) >= 1

## Validation Inspection

Before predicting the leaderboard, we reload the best checkpoint and inspect validation behavior. This protects us from accidentally using the last epoch if it was not the best one.

In [ ]:
model.load_state_dict(torch.load(best_model_path, map_location=device))
model.eval()

val_preds = []
val_gold = []
with torch.no_grad():
    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        logits = model(input_ids, attention_mask)
        val_preds.extend(logits.argmax(dim=1).cpu().numpy())
        val_gold.extend(labels.cpu().numpy())

val_true_categories = [idx2cat[idx] for idx in val_gold]
val_pred_categories = [idx2cat[idx] for idx in val_preds]

print(classification_report(val_true_categories, val_pred_categories, zero_division=0))

## Inference and Submission

The last step mirrors the classical notebooks: predict one category per leaderboard literal, write the CSV, and validate the format.

In [ ]:
test_dataset = ICD10Dataset(lead_df["Literal"].values, labels=None, tokenizer=tokenizer, max_len=MAX_LEN)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

test_pred_indices = []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        with torch.cuda.amp.autocast(enabled=use_amp):
            logits = model(input_ids, attention_mask)
        test_pred_indices.extend(logits.argmax(dim=1).cpu().numpy())

y_lead_pred = [idx2cat[idx] for idx in test_pred_indices]

submission_path = SUBMISSION_DIR / "roberta_baseline_predictions.csv"
submission_df = generate_submission(
    lead_df,
    y_lead_pred,
    output_path=str(submission_path),
)

expected_columns = ["id", "Literal", "y_category"]
assert list(submission_df.columns) == expected_columns
assert len(submission_df) == len(lead_df)
assert submission_df["y_category"].notna().all()
assert (submission_df["y_category"].astype(str).str.len() > 0).all()
assert submission_path.exists()

print("Submission preview:")
display(submission_df.head(10))
print()
print("Category distribution in submission:")
print(submission_df["y_category"].value_counts().sort_index())

## Transformer Takeaways

RoBERTa is the expensive test. If it wins, contextual subword information helped. If it does not, the result still tells us something important: short clinical literals can be dominated by lexical signal.